# R2AI One Click Kaggle Submission

Upload this notebook to Kaggle, enable Internet, select GPU if available, then run all cells. No edits are required.

Output file:

```text
/kaggle/working/r2ai_andgate/results/submission.zip
```


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

REPO_URL = "https://github.com/danhyoyo/r2ai_andgate.git"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/r2ai_andgate")

RUNNER_ARGS = [
    "--install-deps",
    "--import-docs", "5000",
    "--variant", "balanced",
    "--use-dense",
    "--use-cross-encoder",
    "--num-shards", "1",
    "--shard-id", "0",
    "--log-every", "10",
    "--checkpoint-every", "10",
]


def run_cmd(cmd, label):
    print("\n$", " ".join(str(part) for part in cmd), flush=True)
    started = time.time()
    try:
        result = subprocess.run([str(part) for part in cmd], check=False)
    except FileNotFoundError as exc:
        print(f"WARNING: {label} could not start: {exc}", flush=True)
        return False
    minutes = round((time.time() - started) / 60, 2)
    if result.returncode != 0:
        print(f"WARNING: {label} exited with code {result.returncode} after {minutes} minutes", flush=True)
        return False
    print(f"{label} finished in {minutes} minutes", flush=True)
    return True


print("R2AI one-click Kaggle run")
print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Runner args:", " ".join(RUNNER_ARGS))

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run_cmd(["git", "-C", REPO_DIR, "pull", "--ff-only"], "git pull")
elif not REPO_DIR.exists():
    run_cmd(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], "git clone")

if not REPO_DIR.exists():
    raise RuntimeError("Repo directory was not created. Enable Kaggle Internet and rerun the notebook.")

os.chdir(REPO_DIR)
run_cmd(["git", "rev-parse", "--short", "HEAD"], "git revision")

ok = run_cmd([sys.executable, "kaggle_runner.py", *RUNNER_ARGS], "R2AI pipeline")

submission = REPO_DIR / "results" / "submission.zip"
results = REPO_DIR / "results" / "results.json"
print("\nFinal files:")
for path in [results, submission]:
    if path.exists():
        print(f"- {path}: {round(path.stat().st_size / 1024 / 1024, 2)} MB")
    else:
        print(f"- MISSING: {path}")

if not submission.exists():
    raise RuntimeError("submission.zip was not created. Scroll up and check the warning above.")

print("\nDONE. Download this file from Kaggle:")
print(submission)
